# DBSCAN Clustering — Complete Practical Guide

This notebook demonstrates **DBSCAN (Density-Based Spatial Clustering of Applications with Noise)** using a two-moons dataset. It creates data, applies DBSCAN, visualizes clusters, identifies noise, and explores the effect of changing `eps` and `min_samples`.

If the required libraries are missing, run the installation cell first.

## 1. Install required libraries (run only if needed)

In [ ]:
%pip install numpy pandas matplotlib scikit-learn

## 2. Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
from sklearn.datasets import make_moons
from sklearn.neighbors import NearestNeighbors

print('Libraries imported successfully.')

## 3. Create a sample dataset

In [ ]:
# Create 300 data points arranged in two curved moon shapes
X, y_true = make_moons(n_samples=300, noise=0.05, random_state=42)

# Display the original data (true labels are shown only for reference)
plt.figure(figsize=(8, 5))
plt.scatter(X[:, 0], X[:, 1], c=y_true, cmap='viridis', s=35)
plt.title('Original Two-Moons Dataset')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.show()

print('Dataset shape:', X.shape)
print('Number of data points:', len(X))

## 4. Create and run the DBSCAN model

- `eps`: maximum neighborhood distance used to find nearby points.
- `min_samples`: minimum number of points in an `eps` neighborhood for a core point; the point itself is included.
- `fit_predict(X)`: fits DBSCAN to the data and returns a cluster label for each point.
- Label `-1` means DBSCAN classified that point as noise.

In [ ]:
# Create the DBSCAN model
model = DBSCAN(eps=0.2, min_samples=5)

# Fit the model and obtain a cluster label for each point
labels = model.fit_predict(X)

print('Cluster labels found:', np.unique(labels))
print('Number of clusters (excluding noise):', len(set(labels) - {-1}))
print('Number of noise points:', np.sum(labels == -1))

## 5. Visualize the DBSCAN clusters

In [ ]:
plt.figure(figsize=(8, 5))
scatter = plt.scatter(X[:, 0], X[:, 1], c=labels, cmap='rainbow', s=35)
plt.title('DBSCAN Clustering (eps=0.2, min_samples=5)')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.show()

## 6. Inspect cluster assignments in a table

In [ ]:
results = pd.DataFrame({
    'Feature_1': X[:, 0],
    'Feature_2': X[:, 1],
    'Cluster': labels,
    'Actual_Label_For_Reference': y_true
})

# Show the first 10 rows
display(results.head(10))

# Count points assigned to each cluster; -1 represents noise
print('\nPoints per cluster (-1 = noise):')
display(results['Cluster'].value_counts().sort_index().rename_axis('Cluster').to_frame('Number_of_Points'))

## 7. Try different parameter values

A smaller `eps` may mark more points as noise or split clusters. A larger `eps` may merge clusters. The best values depend on the dataset and feature scaling.

In [ ]:
parameter_settings = [
    {'eps': 0.1, 'min_samples': 5},
    {'eps': 0.2, 'min_samples': 5},
    {'eps': 0.3, 'min_samples': 5},
]

fig, axes = plt.subplots(1, len(parameter_settings), figsize=(15, 4.5))

for ax, params in zip(axes, parameter_settings):
    labels_test = DBSCAN(**params).fit_predict(X)
    cluster_count = len(set(labels_test) - {-1})
    noise_count = int(np.sum(labels_test == -1))
    ax.scatter(X[:, 0], X[:, 1], c=labels_test, cmap='rainbow', s=25)
    ax.set_title(
        f"eps={params['eps']}, min_samples={params['min_samples']}\n"
        f"clusters={cluster_count}, noise={noise_count}"
    )
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

plt.tight_layout()
plt.show()

## 8. Optional: K-distance graph to help choose `eps`

The elbow in a sorted k-nearest-neighbor distance plot can help suggest an `eps` value. It is a guide, not a guaranteed automatic answer. Here, `k` is set to 5 to align with `min_samples=5`.

In [ ]:
k = 5
neighbors = NearestNeighbors(n_neighbors=k)
neighbors.fit(X)
distances, indices = neighbors.kneighbors(X)

# Use the distance to the kth nearest neighbor and sort it
k_distances = np.sort(distances[:, -1])

plt.figure(figsize=(8, 5))
plt.plot(k_distances)
plt.title('K-Distance Graph (k=5)')
plt.xlabel('Points sorted by distance')
plt.ylabel('Distance to 5th nearest neighbor')
plt.show()

## 9. Key takeaways

- DBSCAN groups points by density rather than assigning them to centroids.
- You do not need to specify the number of clusters in advance.
- `eps` and `min_samples` strongly affect the result.
- Points labelled `-1` are noise/outliers according to the selected settings.
- For real-world datasets, inspect and scale numeric features when appropriate before applying DBSCAN.